In [ ]:
import pandas as pd

In [ ]:
ds=pd.read_csv('recipes_extended.csv')

In [ ]:
ds.head()

,id,name,ingredients,steps,category,cookingTime,difficulty
0,1,Spaghetti Bolognese,"spaghetti,tomato,beef,onion,garlic",Soğanı kavur; Kıymayı ekle; Domates sosu ekle;...,Dinner,40 dk,Orta
1,2,Caprese Salad,"tomato,mozzarella,basil,olive oil",Domatesleri dilimle; Mozzarella ekle; Zeytinya...,Salad,10 dk,Kolay
2,3,Omelette,"egg,milk,cheese,salt,pepper",Yumurtaları çırp; Tavaya dök; Peyniri ekle; Katla,Breakfast,10 dk,Kolay
3,4,Pancakes,"flour,milk,egg,sugar,butter",Malzemeleri karıştır; Tavada pişir,Breakfast,15 dk,Kolay
4,5,Grilled Cheese Sandwich,"bread,cheese,butter",Ekmeğe peynir koy; Tavada kızart,Snack,5 dk,Kolay


In [ ]:
ds.isnull().sum()

,0
id,0
name,0
ingredients,0
steps,0
category,0
cookingTime,0
difficulty,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


X = ds['ingredients']  # Malzemeler
y = ds['name']       # Yemek isimleri

# TF-IDF için malzemeleri token olarak ayır
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split(','))

X_vec = vectorizer.fit_transform(X)

# Eğitim ve test seti oluştur
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


**MODEL EĞİTİMİ**

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

LogisticRegression(max_iter=500)

In [ ]:
#Modelin test doğruluğu
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test doğruluğu:", accuracy)


Test doğruluğu: 0.0


In [ ]:
def suggest_recipe(ingredients_list):
    text = ",".join(ingredients_list)
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

# Örnek test
print(suggest_recipe(["tomato", "cheese", "basil"]))

Pesto Pasta


In [ ]:
print(suggest_recipe(["tomato"]))
print(suggest_recipe(["egg", "milk", "cheese"]))
print(suggest_recipe(["zucchini","tomato"]))


Tomato Soup
Vegetable Omelette
Tomato Soup


In [ ]:
def suggest_recipes_clean(ingredients_list, top_n=5):
    text = ",".join(ingredients_list)
    vec = vectorizer.transform([text])

    probs = model.predict_proba(vec)[0]  # tüm sınıfların olasılıkları
    classes = model.classes_            # yemek isimleri

    # Sınıfları olasılığa göre sırala
    sorted_indices = probs.argsort()[::-1][:top_n]

    results = []
    for i in sorted_indices:
        name = classes[i]
        probability = round(probs[i] * 100, 2)  # yüzde formatında
        results.append(f"{name} ")

    return results

# Örnek test
print(suggest_recipes_clean(["tomato"]))


['Tomato Soup ', 'Chicken Curry ', 'Tacos ', 'Vegetable Soup ', 'Chicken Salad ']


In [ ]:
print(suggest_recipe(["tomato"]))
print(suggest_recipe(["egg", "milk", "cheese"]))
print(suggest_recipe(["zucchini","tomato"]))

print(suggest_recipes_clean(["tomato"]))
print(suggest_recipes_clean(["egg", "milk", "cheese"]))
print(suggest_recipes_clean(["zucchini","tomato","eggplant"]))



Tomato Soup
Vegetable Omelette
Tomato Soup
['Tomato Soup ', 'Chicken Curry ', 'Tacos ', 'Vegetable Soup ', 'Chicken Salad ']
['Vegetable Omelette ', 'Omelette ', 'Mac and Cheese ', 'Pancakes ', 'French Toast ']
['Tomato Soup ', 'Chicken Curry ', 'Tacos ', 'Vegetable Soup ', 'Chicken Salad ']


**FLUTTERA BAĞLANMAK İÇİN FLASK API OLUŞTURUYORUZ**


FastApı veya flask ile oluşturuluyor
-FastApi: Daha büyük modeller için oluşturuluyor.
-Flask: Deneme amacıyla kullanılıyor. Asenkron programlama desteği sınırlıdır.

In [ ]:
!pip install flask pyngrok nest-asyncio

from flask import Flask, request, jsonify
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Flask app
app = Flask(__name__)

@app.route("/suggest", methods=["POST"])
def suggest():
    data = request.json
    ingredients = data.get("ingredients", [])
    top_n = data.get("top_n", 3)

    suggestions = suggest_recipes_clean(ingredients, top_n=top_n)
    return jsonify({"suggestions": suggestions})

# ngrok authtoken'ınızı buraya yapıştırın
ngrok.set_auth_token("37ZskcvGuQKEHxNguHvnznL2kJj_ZbHJZWYPVkJ5oU34JGsU")

# Colab üzerinde public URL oluştur
public_url = ngrok.connect(5000)
print("API URL:", public_url)

# Flask çalıştır
app.run(port=5000)


API URL: NgrokTunnel: "https://tritely-prediligent-lilianna.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Jan/2026 11:59:24] "POST /suggest HTTP/1.1" 200 -
